In [2]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Feature Engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

#Load Dataset
df = pd.read_csv("twitter_training.csv", header=None)

df.columns = ['ID', 'Entity', 'Sentiment', 'Text']

# Remove missing text rows
df = df.dropna(subset=['Text'])

print(df.shape)
print(df['Sentiment'].value_counts())

print("Shape:", df.shape)
print("\nColumns:\n", df.columns)

# Check missing values
print("\nMissing Values:\n", df.isnull().sum())

# Class distribution
print("\nClass Distribution:\n", df['Sentiment'].value_counts())

# Sample text
print("\nSample Text:\n", df['Text'].iloc[0])

df.columns = ['ID', 'Entity', 'Sentiment', 'Text']

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove special characters & punctuation
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenization
    words = text.split()

    # Remove stopwords + stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]

    return " ".join(words)

df['clean_text'] = df['Text'].astype(str).apply(preprocess_text)

df[['Text', 'clean_text']].head()

bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text']).toarray()

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

y = df['Sentiment']

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)


lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

def evaluate_model(name, y_test, y_pred):
    print(f"\n{name} Performance:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Naive Bayes", y_test, y_pred_nb)
evaluate_model("Decision Tree", y_test, y_pred_dt)

print("\nLogistic Regression Report:\n", classification_report(y_test, y_pred_lr))
print("\nNaive Bayes Report:\n", classification_report(y_test, y_pred_nb))
print("\nDecision Tree Report:\n", classification_report(y_test, y_pred_dt))



# Observations & Insights
# Best Preprocessing Step:
# Removing stopwords and stemming improved model performance by reducing noise.
# Best Feature Engineering:
# TF-IDF performed better than Bag of Words because it considers word importance.
# Best Model:
# Logistic Regression gave the highest accuracy and F1-score.
# Naive Bayes:
# Fast but slightly less accurate.
# Decision Tree:
# Overfitting observed, less stable.

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


(73996, 4)
Sentiment
Negative      22358
Positive      20655
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64
Shape: (73996, 4)

Columns:
 Index(['ID', 'Entity', 'Sentiment', 'Text'], dtype='object')

Missing Values:
 ID           0
Entity       0
Sentiment    0
Text         0
dtype: int64

Class Distribution:
 Sentiment
Negative      22358
Positive      20655
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64

Sample Text:
 im getting on borderlands and i will murder you all ,

Logistic Regression Performance:
Accuracy: 0.6860135135135135
Precision: 0.6870968050978037
Recall: 0.6860135135135135
F1 Score: 0.6820844463258752

Naive Bayes Performance:
Accuracy: 0.6372297297297297
Precision: 0.6382897173034443
Recall: 0.6372297297297297
F1 Score: 0.6314607913207795

Decision Tree Performance:
Accuracy: 0.7971621621621622
Precision: 0.7985234090534704
Recall: 0.7971621621621622
F1 Score: 0.7968237960356196

Logistic Regression Report:
               precisio